# Downstream Evaluation: RankMe Evolution → Multilingual Performance

**Research question:** *Does the shape of the RankMe trajectory during pre-training predict downstream multilingual performance? Specifically: does the point where compression begins (onset) and how long it lasts (duration) correlate with accuracy on m-MMLU and XCOPA?*

**Models:** FuxiTranyu-8B · Apertus-8B-2509  
**Benchmarks:** m-MMLU (5-shot, 8 languages) · XCOPA (0-shot, 5 languages)

### What this notebook does
1. **Phase identification** — detects two training phases from the RankMe curve: an *entropy-seeking* phase (RankMe ↑, the model builds richer representations) and a *compression-seeking* phase (RankMe ↓, the model specializes and compresses). For each phase, **onset** is the token count (in billions) where the phase begins, and **duration** is how many tokens it lasts. Works with `fuxi_fine_wiki.csv` alone.
2. **Grokking detection** — finds the first checkpoint where accuracy rises more than 15 percentage points above random-chance and stays there for at least 2 consecutive checkpoints. This marks the point where the model transitions from near-random guessing to genuinely solving the task. Requires downstream eval results.
3. **Correlation analysis** — Spearman + Pearson tests: *does an earlier or longer compression phase predict earlier or higher accuracy?* Computed separately per task (m-MMLU / XCOPA), never pooled across tasks.
4. **Plots** — RankMe trajectories, phase overlays (shaded regions drawn on the RankMe curve that visually mark where each phase occurs), dual-axis accuracy curves, and a correlation scatter per task.

> **Note:** Cells that require downstream evaluation results degrade gracefully if `results/eval/` is empty. Run `evaluate.py` (or `submit_eval.sh`) first, then rerun this notebook.

In [ ]:
import warnings

import pandas as pd

from utils import (
    load_config,
    load_rankme_data, load_eval_data,
    compute_phases, compute_alpha_phases,
    compute_grokking,
    compute_correlations_table, compute_alpha_correlations_table,
    plot_rankme_phases, plot_overlay,
    plot_alpha_phases, plot_alpha_overlay,
    plot_rankme_last_scatter_combined, plot_correlation_scatter_combined,
    plot_alpha_correlation_scatter_combined, plot_alpha_rate_scatter_combined,
)

warnings.filterwarnings("ignore")
print("Imports OK")

## Configuration

The notebook analyses **both models simultaneously**. `load_config()` derives all paths, layer settings, grokking parameters, and benchmark metadata from `configs/benchmarks.yaml`.

Per-model plots (trajectories, phase overlays) are saved with the model name in the filename (e.g. `rankme_phases_fuxi.png`, `overlay_m_mmlu_apertus.png`). Combined scatter plots overlay both models in a single file.

In [ ]:
MODELS = ["fuxi", "apertus"]

# Per-model colors and markers used in combined scatter plots.
_PALETTE = {
    "fuxi":    {"color": "steelblue",  "marker": "o"},
    "apertus": {"color": "darkorange", "marker": "s"},
}

data = {}
for m in MODELS:
    cfg = load_config(m)
    data[m] = {
        "cfg":    cfg,
        "color":  _PALETTE[m]["color"],
        "marker": _PALETTE[m]["marker"],
    }
    print(f"[{m}] {cfg['model_label']}  |  Layer: {cfg['layer']}  |  Aggregation: {cfg['aggregation']}")

## Data

Two separate sources are loaded:

- **`fuxi_fine_wiki.csv` / `apertus_fine_wiki.csv`** — RankMe geometry metrics produced by `geometry_analysis.py` for all model checkpoints, all languages, and all layers. Used for phase identification and phase plots. The working slice (`layer_29`, `agg=last`) gives one RankMe value per (checkpoint, language) pair across the full training run.

- **`fuxi_fine_wiki_merged.csv` / `apertus_fine_wiki_merged.csv`** — downstream accuracy merged with geometry metrics, produced by `merge_results.py` after running evaluation on the cluster. Only contains the checkpoints that have been evaluated. Used for grokking detection, correlation analysis, and overlay plots.

The phase plots use the full geometry data (all checkpoints). The accuracy-based plots use only the evaluated checkpoints.

In [ ]:
for m in MODELS:
    cfg = data[m]["cfg"]
    df_rankme, df_layer, checkpoints_all, token_counts, langs_sorted = load_rankme_data(
        cfg["rankme_csv"], cfg["layer"], cfg["aggregation"])
    df_eval, eval_available = load_eval_data(cfg["merged_csv"])
    data[m].update({
        "df_rankme":       df_rankme,
        "df_layer":        df_layer,
        "checkpoints_all": checkpoints_all,
        "token_counts":    token_counts,
        "langs_sorted":    langs_sorted,
        "df_eval":         df_eval,
        "eval_available":  eval_available,
    })
    print()

## Phase Identification

The RankMe curve is split into two phases by locating its global peak:

- **Entropy-seeking** — from the first observed checkpoint to the peak (RankMe ↑). The model is building richer, more diverse representations.
- **Compression-seeking** — from the peak to the end of training (RankMe ↓). The model specialises and compresses its representations.

For each phase, **onset** is the token count where it begins and **duration** is how long it lasts. All token values are in **billions (B)**.

A phase is `NaN` when it cannot be observed in the available data:
- **Entropy NaN** — the peak is at the very first checkpoint we have, meaning the entropy-seeking phase either happened before training began or before the earliest checkpoint recorded. This is the case for FuxiTranyu-8B, where RankMe is highest at 10B tokens and decreases from there.
- **Compression NaN** — RankMe never falls after the peak, so no compression phase is detected.

In [ ]:
for m in MODELS:
    d = data[m]
    cfg = d["cfg"]
    df_phases = compute_phases(d["df_layer"], d["checkpoints_all"], d["token_counts"])
    d["df_phases"] = df_phases

    display_cols = {
        "language":                    "language",
        "peak_tokens":                 "peak (B)",
        "entropy_onset_tokens":        "entropy onset (B)",
        "entropy_duration_tokens":     "entropy duration (B)",
        "compression_onset_tokens":    "compression onset (B)",
        "compression_duration_tokens": "compression duration (B)",
        "rankme_first":                "RankMe first ckpt",
        "rankme_last":                 "RankMe last ckpt",
        "rankme_decline_rate":         "decline rate (1/B)",
    }
    print(f"\n── {cfg['model_label']} ────────────────────────────────────────")
    print(df_phases[list(display_cols)].rename(columns=display_cols).to_string(index=False))

In [ ]:
for m in MODELS:
    d = data[m]
    cfg = d["cfg"]
    plot_rankme_phases(d["df_layer"], d["df_phases"], d["checkpoints_all"], d["token_counts"],
                       d["langs_sorted"], cfg["model_label"], cfg["layer"], cfg["aggregation"],
                       cfg["plots_dir"], model_key=m)

## Grokking Detection

For each (task, language) pair, the table below reports when the model transitions from near-random guessing to genuinely solving the task:

- **grokking_tokens (B)** — token count (in billions) at the first checkpoint where accuracy exceeds random chance by more than 15 percentage points and stays there for at least 2 consecutive checkpoints.
- **peak_tokens (B)** — token count at the checkpoint with the highest observed accuracy.
- **peak_accuracy** — highest accuracy seen across all evaluated checkpoints.
- **random_chance** — baseline accuracy for a random classifier (0.25 for 4-way m-MMLU, 0.50 for 2-way XCOPA).

A `NaN` grokking onset means one of two things: either the model never exceeded the threshold at any evaluated checkpoint (accuracy stays near random chance throughout), or the threshold was crossed only once without the required 2 consecutive checkpoints.

In [ ]:
for m in MODELS:
    d = data[m]
    cfg = d["cfg"]
    if d["eval_available"]:
        df_grokking = compute_grokking(d["df_eval"], cfg["task_languages"], cfg["random_chance"],
                                       threshold=cfg["grokking_threshold"],
                                       min_consecutive=cfg["grokking_min_consec"])
        d["df_grokking"] = df_grokking
        display_cols = {
            "task":            "task",
            "language":        "language",
            "grokking_tokens": "grokking onset (B)",
            "peak_tokens":     "peak (B)",
            "peak_accuracy":   "peak accuracy",
            "random_chance":   "random chance",
        }
        print(f"\n── {cfg['model_label']} ────────────────────────────────────────")
        print(df_grokking[list(display_cols)].rename(columns=display_cols).to_string(index=False))
    else:
        d["df_grokking"] = pd.DataFrame()
        print(f"[{m}] No eval data — run submit_eval.sh + merge_results.py first.")

## Correlation Analysis

Tests whether RankMe phase geometry predicts downstream performance. Five predictors are tested against two outcomes, separately per task:

**Timing-based (may have zero variance if all languages peak at the same checkpoint):**
- **Compression onset (B)** — how early the compression phase begins.
- **Compression duration (B)** — how long the compression phase lasts.

**Magnitude-based (vary across languages regardless of when the peak occurs):**
- **RankMe at first ckpt** — representation richness at the start of training.
- **RankMe at last ckpt** — representation richness at the end of training.
- **Rate of RankMe decline (1/B)** — (RankMe[first] − RankMe[last]) / total training tokens; how aggressively the model compresses each language's representations.

Outcomes:
- **Grokking onset (B)** — when accuracy first crosses the threshold.
- **Peak accuracy** — highest accuracy across all evaluated checkpoints.

Both Spearman (rank-based) and Pearson (linear) correlations are reported, computed separately per task and never pooled.

In [ ]:
for m in MODELS:
    d = data[m]
    cfg = d["cfg"]
    if d["eval_available"] and not d["df_grokking"].empty:
        df_correlations = compute_correlations_table(d["df_grokking"], d["df_phases"])
        d["df_correlations"] = df_correlations
        print(f"\n── {cfg['model_label']} ────────────────────────────────────────")
        print(df_correlations.to_string(index=False))
    else:
        d["df_correlations"] = pd.DataFrame()
        print(f"[{m}] Skipping correlation analysis — no eval data available.")

The scatter plot below visualises compression onset vs peak accuracy (one panel per task) with labeled points per language and a linear regression line.

In [ ]:
models_data = [
    {
        "label":       data[m]["cfg"]["model_label"],
        "df_grokking": data[m].get("df_grokking", pd.DataFrame()),
        "df_phases":   data[m].get("df_phases",   pd.DataFrame()),
        "color":       data[m]["color"],
        "marker":      data[m]["marker"],
    }
    for m in MODELS if data[m]["eval_available"]
]
if models_data:
    plot_correlation_scatter_combined(models_data, data[MODELS[0]]["cfg"]["plots_dir"])
else:
    print("[INFO] Skipping correlation scatter — no eval data available.")

🔑 **Key finding:** RankMe at the last checkpoint predicts peak XCOPA accuracy: **Spearman r=0.90, p=0.037** (n=5, significant). Languages the model maintains richer representations for throughout training also achieve higher commonsense accuracy — the absolute level of representational richness matters more than when or how fast it declines.

In [ ]:
if models_data:
    plot_rankme_last_scatter_combined(models_data, data[MODELS[0]]["cfg"]["plots_dir"])
else:
    print("[INFO] Skipping RankMe last checkpoint scatter — no eval data available.")

## Overlay Plots

One plot per task (m-MMLU, XCOPA), with one panel per language that appears in both the RankMe data and the evaluation results. Each panel shows two y-axes:

- **Left axis (blue)** — RankMe trajectory across all checkpoints, showing the full geometry evolution during pre-training.
- **Right axis (red, dashed)** — downstream accuracy at the evaluated checkpoints only. The axis always spans 0 to 1.

The shaded regions mark the two training phases (green = entropy-seeking, orange = compression-seeking). The black dashed horizontal line marks the random-chance baseline (0.25 for m-MMLU, 0.50 for XCOPA). A vertical purple dashed line marks the grokking onset for that language — it is absent when grokking was not detected (accuracy never exceeded the threshold for 2 consecutive checkpoints).

These plots let you visually inspect whether accuracy rises during or after the compression phase, and whether grokking coincides with any particular point in the RankMe trajectory.

In [ ]:
for m in MODELS:
    d = data[m]
    cfg = d["cfg"]
    if d["eval_available"]:
        plot_overlay(d["df_eval"], d["df_layer"], d["df_phases"], d["df_grokking"],
                     cfg["task_languages"], cfg["random_chance"],
                     d["checkpoints_all"], d["token_counts"], d["langs_sorted"],
                     cfg["model_label"], cfg["plots_dir"], model_key=m)
    else:
        print(f"[{m}] Skipping overlay plots — run evaluate.py first.")

## RankMe Findings

### FuxiTranyu-8B

**Phase structure (RankMe, all 13 languages)**

- RankMe peaks at the very first checkpoint (10B tokens) and declines throughout training for every language — the compression phase onset is synchronised across all languages. Because there is zero variance in the onset, timing-based correlations (compression onset, duration) are all undefined.
- The magnitude of RankMe varies substantially across languages both at the start (169–577) and at the end (44–278) of training, providing a meaningful cross-language predictor even when timing is uniform.

**m-MMLU knowledge accuracy (8 languages)**

- Accuracy remains at random chance (~25%) for all 8 languages across all evaluated checkpoints — no language ever exceeds the 40% grokking threshold. No grokking is detected.
- The tiny spread in peak accuracy (0.269–0.276, i.e. 0.7 pp) is within statistical noise; no RankMe metric shows a significant correlation with m-MMLU performance.
- This confirms that factual knowledge retrieval (m-MMLU) does not emerge during pre-training alone, regardless of language or representation geometry.

**XCOPA commonsense accuracy (5 languages)**

- All languages exceed the 50% random-chance baseline from the first checkpoint, indicating commonsense reasoning ability emerges very early in training.
- Accuracy improves monotonically throughout training — no regression or forgetting is observed at any checkpoint.
- The performance ranking (Vietnamese > Italian > Indonesian > Chinese > Turkish) does not match training data composition. A more plausible explanation is cross-lingual transfer from Latin-script languages: Vietnamese, Italian, and Indonesian all use Latin script and likely benefit from English transfer, while Turkish agglutinative morphology and Chinese script may limit transfer.
- Grokking was detected in 4 of 5 languages (Vietnamese at 126B, Italian and Indonesian at 241B, Chinese at 325B). Turkish accuracy improved too gradually to trigger the threshold.

**Correlation analysis (RankMe magnitude → XCOPA)**

- **RankMe at the last checkpoint predicts peak XCOPA accuracy: Spearman r=0.90, p=0.037 (n=5, significant).** Languages the model maintains richer representations for throughout training also achieve higher commonsense accuracy — the absolute level of representational richness matters more than when or how fast it declines.
- RankMe at the last checkpoint also predicts grokking onset (Pearson r=−0.97, p=0.03): languages with higher final RankMe grok earlier.
- The rate of RankMe decline shows no relationship with either outcome (r≈0, p≈1), confirming it is the richness level — not the compression speed — that drives downstream performance.
- These results should be interpreted with caution given the small sample (n=5 languages for XCOPA).

---

### Apertus-8B-2509

> **Placeholder** — Apertus findings to be written after reviewing the combined scatter plots and per-model trajectory/overlay plots above.

Key questions to address:
- Does RankMe also peak at the first checkpoint, or is the entropy-seeking phase visible?
- Does m-MMLU accuracy remain at random chance, or does Apertus show factual knowledge emergence?
- Which languages grok on XCOPA, and at what token counts?
- Does RankMe at the last checkpoint predict XCOPA accuracy for Apertus as it does for Fuxi?

---
## Extended Analysis: AlphaReQ

AlphaReQ (α) measures the power-law exponent of the eigenvalue spectrum of the activation matrix. Unlike RankMe, which captures *how many* dimensions are used, α captures the *shape* of the spectrum:

- **Lower α** → heavier-tailed spectrum → more implicit self-regularization → associated with better generalisation.
- **Higher α** → lighter-tailed spectrum → less regularization.

The **α trough** is the training checkpoint where α reaches its global minimum — the point of maximum self-regularization. It marks the boundary between two phases we define for this analysis:

- **Regularization-increasing** (α ↓): α falls from the first checkpoint to the trough — the model is gaining self-regularization.
- **Regularization-decreasing** (α ↑): α rises from the trough to the end of training — self-regularization weakens slightly as the model continues to adapt.

Unlike the RankMe analysis where the peak was at 10B for every language, the α trough position **varies across languages** (189B–531B), making the correlation analysis more informative.

### Phase Identification (AlphaReQ)

In [ ]:
for m in MODELS:
    d = data[m]
    cfg = d["cfg"]
    df_alpha_phases = compute_alpha_phases(d["df_layer"], d["checkpoints_all"], d["token_counts"])
    d["df_alpha_phases"] = df_alpha_phases

    display_cols = {
        "language":                       "language",
        "trough_tokens":                  "trough (B)",
        "reg_increasing_onset_tokens":    "reg-increasing onset (B)",
        "reg_increasing_rate":            "rate of α decline (α/B)",
        "reg_decreasing_onset_tokens":    "reg-decreasing onset (B)",
        "reg_decreasing_duration_tokens": "reg-decreasing duration (B)",
    }
    print(f"\n── {cfg['model_label']} ────────────────────────────────────────")
    print(df_alpha_phases[list(display_cols)].rename(columns=display_cols).to_string(index=False))

In [ ]:
for m in MODELS:
    d = data[m]
    cfg = d["cfg"]
    plot_alpha_phases(d["df_layer"], d["df_alpha_phases"], d["checkpoints_all"], d["token_counts"],
                      d["langs_sorted"], cfg["model_label"], cfg["layer"], cfg["aggregation"],
                      cfg["plots_dir"], model_key=m)

### Overlay Plots (AlphaReQ)

Each panel shows AlphaReQ (blue, left axis) overlaid with downstream accuracy (red, right axis). Shaded regions mark the two α phases (blue = regularization-increasing, purple = regularization-decreasing). The black dashed line marks random-chance baseline. The purple vertical dashed line marks grokking onset where detected.

In [ ]:
for m in MODELS:
    d = data[m]
    cfg = d["cfg"]
    if d["eval_available"]:
        plot_alpha_overlay(d["df_eval"], d["df_layer"], d["df_alpha_phases"], d["df_grokking"],
                           cfg["task_languages"], cfg["random_chance"],
                           d["checkpoints_all"], d["token_counts"], d["langs_sorted"],
                           cfg["model_label"], cfg["plots_dir"], model_key=m)
    else:
        print(f"[{m}] Skipping AlphaReQ overlay plots — run evaluate.py first.")

### Correlation Analysis (AlphaReQ)

Tests whether the position of the α trough (onset of the regularization-decreasing phase) or the rate of α decline (drop in α per billion tokens during the regularization-increasing phase) predicts downstream performance. Unlike the RankMe analysis where all languages shared the same compression onset, the α trough varies across languages (189B–531B), so correlations here carry real predictive signal.

In [ ]:
for m in MODELS:
    d = data[m]
    cfg = d["cfg"]
    if d["eval_available"] and not d["df_grokking"].empty:
        df_alpha_correlations = compute_alpha_correlations_table(d["df_grokking"], d["df_alpha_phases"])
        d["df_alpha_correlations"] = df_alpha_correlations
        print(f"\n── {cfg['model_label']} ────────────────────────────────────────")
        print(df_alpha_correlations.to_string(index=False))
    else:
        d["df_alpha_correlations"] = pd.DataFrame()
        print(f"[{m}] Skipping AlphaReQ correlation analysis — no eval data available.")

In [ ]:
alpha_models_data = [
    {
        "label":           data[m]["cfg"]["model_label"],
        "df_grokking":     data[m].get("df_grokking",     pd.DataFrame()),
        "df_alpha_phases": data[m].get("df_alpha_phases", pd.DataFrame()),
        "color":           data[m]["color"],
        "marker":          data[m]["marker"],
    }
    for m in MODELS if data[m]["eval_available"]
]
if alpha_models_data:
    plot_alpha_correlation_scatter_combined(alpha_models_data, data[MODELS[0]]["cfg"]["plots_dir"])
else:
    print("[INFO] Skipping AlphaReQ correlation scatter — no eval data available.")

🔑 **Key finding:** The rate of α decline is the strongest predictor of grokking onset: **Spearman r=−0.95 (p=0.05), Pearson r=−0.98 (p=0.02)**. Languages where α drops faster during training grok earlier — interpreted with caution given n=4.

In [ ]:
if alpha_models_data:
    plot_alpha_rate_scatter_combined(alpha_models_data, data[MODELS[0]]["cfg"]["plots_dir"])
else:
    print("[INFO] Skipping AlphaReQ rate scatter — no eval data available.")

### AlphaReQ Findings

#### FuxiTranyu-8B

**Phase structure (AlphaReQ, all 13 languages)**

- Both phases are present in every language: α falls to a trough then rises slightly, confirming the regularization-increasing → regularization-decreasing pattern is universal.
- The α trough position **varies substantially across languages** (189B–531B), in sharp contrast to RankMe where all languages peaked at 10B. Latin-script languages reach the trough early (~189–241B), while Chinese and Swahili reach it much later (~531B), suggesting these languages require more training tokens to achieve maximum self-regularization.

**Relationship with downstream accuracy (XCOPA)**

- Visually, the overlay plots suggest that α declines during the same training window where XCOPA accuracy rises, but we do not directly measure the relationship between the rate of α decline and the rate of accuracy rise. This would require derivative estimation on noisy trajectories and is left for future analysis with more languages.
- The correlation between the trough position and grokking onset is strong (Spearman r=0.83) but not statistically significant with only 4 valid pairs (p=0.17). Languages that take longer to reach peak regularization also tend to grok later — an intuitive result.
> The **rate of α decline** is a stronger predictor of grokking onset: Spearman r=−0.95 (p=0.05) and Pearson r=−0.98 (p=0.02). Languages where α drops faster during training grok earlier. This is the strongest correlation observed in the analysis, though the small sample (4 languages) means it should be interpreted with caution.
- No meaningful correlation between trough position or rate of α decline and peak accuracy is observed for XCOPA alone. With m-MMLU data across more languages, this analysis will be more conclusive.

---

#### Apertus-8B-2509

> **Placeholder** — Apertus AlphaReQ findings to be written after reviewing the combined scatter plots and per-model trajectory/overlay plots above.

Key questions to address:
- Does the α trough position vary across languages for Apertus, or is it synchronised as with Fuxi's RankMe peak?
- Is the rate of α decline a similarly strong predictor of grokking onset for Apertus?
- Do the regularization phases align with accuracy emergence in the overlay plots?


### Workflow
```bash
# 1 — Submit one cluster job per checkpoint (see README)
for ckpt in 10B 115B ...; do
    ./downstream_evaluation/submit_eval.sh $ckpt fuxi
done

# 2 — After all jobs complete, merge once manually (see README)

# 3 — Rerun this notebook — all cells activate automatically
```